# rift `nisar2cog` — DPS job runner (bounding-box driven)

Query ASF for NISAR **GSLC** granules over an area of interest, filter to the **40 MHz
frequency-A (40+5 MHz)** mode, and submit one MAAP DPS job per granule to the
`rift-nisar2cog` OGC process. Start on the **`maap-dps-sandbox`** queue.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `rift-nisar2cog` OGC process is deployed (via `.github/workflows/ogc-app-pack.yml`).
- `asf-search` installed (`pip install asf-search`).


In [ ]:
# One-time in a fresh workspace:
# %pip install asf-search
import asf_search as asf
import pandas as pd
import datetime, json, os, time
from maap.maap import MAAP

maap = MAAP()
print("asf-search", asf.__version__)

## 1. Area of interest

Thwaites / Pine Island sector. We pass the WKT polygon straight to `intersectsWith=` for
precise spatial filtering (more accurate than a bbox).

In [ ]:
AOI_WKT = ("POLYGON((-101.9351 -74.7848,-102.4704 -75.4215,"
           "-99.852 -75.5511,-99.4242 -74.9087,-101.9351 -74.7848))")

# Equivalent bounding box (MINX MINY MAXX MAXY), for reference / STAC fallbacks:
AOI_BBOX = "-102.4704 -75.5511 -99.4242 -74.7848"
AOI_WKT

## 2. Search ASF for NISAR GSLC granules

`asf-search` exposes NISAR datasets once they are public. Depending on the installed
version the dataset/collection identifiers may differ, so we try a couple of spellings and
keep whichever returns results. Inspect one `.properties` dict (next cell) to learn the
exact fields available for the 40 MHz filter.

In [ ]:
def search_gslc(wkt):
    attempts = [
        dict(dataset=getattr(asf, "DATASET", object).__dict__.get("NISAR", "NISAR"),
             processingLevel="GSLC"),
        dict(platform="NISAR", processingLevel="GSLC"),
        dict(processingLevel="GSLC"),
    ]
    last_err = None
    for kw in attempts:
        try:
            res = asf.geo_search(intersectsWith=wkt, maxResults=200, **kw)
            if len(res):
                print(f"matched {len(res)} granules with kwargs={kw}")
                return res
        except Exception as e:  # noqa: BLE001
            last_err = e
            print(f"  search kwargs={kw} failed: {e}")
    if last_err:
        print("all attempts failed; last error above")
    return asf.ASFSearchResults([])

results = search_gslc(AOI_WKT)
len(results)

In [ ]:
# Inspect the properties of one result to discover the exact fields for the 40 MHz filter
# (e.g. bandwidth / mode / rangeBandwidth / productType). Adjust FILTER below to match.
if len(results):
    print(json.dumps(results[0].properties, indent=2, default=str))
else:
    print("No results — verify NISAR GSLC is public in this asf-search version, or adjust kwargs.")

## 3. Filter to 40 MHz frequency-A (40+5 mode)

NISAR's science modes pair a wide **40 MHz** frequency-A channel with a narrow 5 MHz
frequency-B channel ("40+5"). The exact property name varies by `asf-search` version —
after inspecting `.properties` above, set `FIELD` / matching logic here. This cell **logs
how many granules were dropped** so nothing is silently truncated.

In [ ]:
def is_40mhz(props):
    """Best-effort 40 MHz freq-A check. Tune to the real property names printed above."""
    hay = " ".join(str(props.get(k, "")).lower()
                    for k in ("beamModeType", "beamMode", "configurationName",
                              "bandwidth", "rangeBandwidth", "productType", "sceneName",
                              "fileName", "granuleName"))
    # Common encodings of the wide-band mode: "40", "40+5", "40mhz".
    return ("40+5" in hay) or ("40mhz" in hay) or (" 40 " in f" {hay} ") or ("40_05" in hay)

if len(results):
    kept = [r for r in results if is_40mhz(r.properties)]
    dropped = len(results) - len(kept)
    print(f"kept {len(kept)} / {len(results)} granules (dropped {dropped} not matching 40 MHz freq-A)")
    if dropped and not kept:
        print("WARNING: filter removed everything — inspect .properties and adjust is_40mhz().")
else:
    kept = []
kept_urls = [r.properties.get("url") for r in kept if r.properties.get("url")]
kept_urls[:5]

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"id":..., "title":...}, ...]}`. Find the
`rift-nisar2cog` entry and grab its `id`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("id"), "|", p.get("title"))

PROCESS_ID = next((p["id"] for p in procs
                   if "nisar2cog" in str(p.get("id", "")).lower()
                   or "nisar2cog" in str(p.get("title", "")).lower()), None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`kept_urls[:1]`) on `maap-dps-sandbox` to validate before
scaling. `submit_job` returns HTTP 202 with `{"id": ..., "status": "accepted"}`.

In [ ]:
QUEUE = "maap-dps-sandbox"        # 8 GB, 10-min cap. Fall back to maap-dps-worker-16gb/-32gb.
TAG = "nisar2cog-thwaites"
POLS = ""                          # empty = all freq-A pols
AMP_ONLY = "false"

TEST_URLS = kept_urls[:1]          # <-- widen to kept_urls once the first job succeeds
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed?"
assert TEST_URLS, "No granule URLs — check the search/filter cells."

rows = []
for i, url in enumerate(TEST_URLS, start=1):
    inputs = {"gslc_url": url, "pols": POLS, "amp_only": AMP_ONLY}
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(TEST_URLS)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "gslc_url": url, "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/nisar2cog_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/rift-nisar2cog/...`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    # metrics (OGC v5 only):
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))